<p style="text-align:right"><img src="https://www.iesb.br/content/themes/iesb-chleba-themosis/assets/imagens/logo.png" width="100"/></p>

# 🧠 Laboratório 9 — Agente com ferramenta

**Inteligência Artificial (MDC050)** · Centro Universitário IESB · Turma ENGCNM2B
**Prof. Rodrigo Gonçalves Pinto** · Aula 10 · 28/10/2026

---

## O que você vai fazer aqui

No Laboratório 8, o **código** decidia a ordem: extrair, conferir, chamar o A\*. Hoje é o **modelo** que decide se precisa de uma ferramenta, qual ferramenta usar e com quais dados. Depois ele recebe o resultado e explica ao usuário.

| Etapa | O que acontece |
|---|---|
| 0 | Preparar o Colab e carregar o Qwen |
| 1 | O mapa, o A\* e as duas ferramentas |
| 2 | Ver o modelo pedindo uma ferramenta |
| 3 | Ler o pedido de ferramenta e executar |
| 4 | O ciclo completo do agente |
| 5 | Testar a escolha da ferramenta |
| Desafio | A ferramenta do seu trio |

## Como entregar

1. Complete os **seis TODOs**. Embaixo de cada um há uma verificação que imprime ✅ ou ❌.
2. Responda a célula **Sua análise**, no final.
3. **Antes de entregar:** `Ambiente de execução → Reiniciar sessão e executar tudo`.
4. Envie o link do notebook pelo Google Sala de Aula.

> **Crie sua cópia antes de começar:** `Arquivo → Salvar uma cópia no Drive`.

---
# Etapa 0 — Preparação

**GPU:** `Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU → Salvar`. Sem GPU disponível, escolha CPU: funciona, só fica mais lento.

**Token do Hugging Face** (se ainda não criou): huggingface.co → foto → **Settings** → **Access Tokens** → **Create new token** → tipo **Read** → copie. No Colab: ícone da **chave 🔑** → **Adicionar novo secret** → nome `HF_TOKEN` → cole o token → ative **Acesso ao notebook**. Se não conseguir, pule a célula de login: o modelo baixa mesmo assim.

In [ ]:
import torch
import json

if torch.cuda.is_available():
    dispositivo = "cuda"
    print("✅ GPU ligada.")
else:
    dispositivo = "cpu"
    print("⚠️ Sem GPU. Funciona, mas as respostas vão demorar mais.")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("✅ Login no Hugging Face feito.")

## Carregar o Qwen

A função `conversar` agora recebe também a lista de **ferramentas** que o modelo pode usar. O resto é igual ao Laboratório 8.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

nome_qwen = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizador_qwen = AutoTokenizer.from_pretrained(nome_qwen)

if dispositivo == "cuda":
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float16).to(dispositivo)
else:
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float32)
qwen.eval()


def conversar(mensagens, ferramentas=None, max_tokens=150):
    entrada = tokenizador_qwen.apply_chat_template(
        mensagens, tools=ferramentas, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(dispositivo)
    with torch.no_grad():
        saida = qwen.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    tamanho_da_pergunta = entrada["input_ids"].shape[-1]
    return tokenizador_qwen.decode(saida[0][tamanho_da_pergunta:], skip_special_tokens=True)


print("✅ Qwen carregado em", dispositivo)

---
# Etapa 1 — O mapa, o A\* e as ferramentas

## O mapa e o A\* (iguais ao Laboratório 8)

In [ ]:
CIDADES = ["Arad", "Zerind", "Oradea", "Sibiu", "Timisoara", "Lugoj", "Mehadia",
           "Drobeta", "Craiova", "Rimnicu Vilcea", "Fagaras", "Pitesti", "Bucareste",
           "Giurgiu", "Urziceni", "Hirsova", "Eforie", "Vaslui", "Iasi", "Neamt"]

ESTRADAS = {
    ("Arad", "Zerind"): 75, ("Arad", "Sibiu"): 140, ("Arad", "Timisoara"): 118,
    ("Zerind", "Oradea"): 71, ("Oradea", "Sibiu"): 151, ("Timisoara", "Lugoj"): 111,
    ("Lugoj", "Mehadia"): 70, ("Mehadia", "Drobeta"): 75, ("Drobeta", "Craiova"): 120,
    ("Craiova", "Rimnicu Vilcea"): 146, ("Craiova", "Pitesti"): 138, ("Sibiu", "Fagaras"): 99,
    ("Sibiu", "Rimnicu Vilcea"): 80, ("Rimnicu Vilcea", "Pitesti"): 97, ("Fagaras", "Bucareste"): 211,
    ("Pitesti", "Bucareste"): 101, ("Bucareste", "Giurgiu"): 90, ("Bucareste", "Urziceni"): 85,
    ("Urziceni", "Hirsova"): 98, ("Hirsova", "Eforie"): 86, ("Urziceni", "Vaslui"): 142,
    ("Vaslui", "Iasi"): 92, ("Iasi", "Neamt"): 87,
}
LINHA_RETA_ATE_BUCARESTE = {
    "Arad": 366, "Bucareste": 0, "Craiova": 160, "Drobeta": 242, "Eforie": 161, "Fagaras": 176,
    "Giurgiu": 77, "Hirsova": 151, "Iasi": 226, "Lugoj": 244, "Mehadia": 241, "Neamt": 234,
    "Oradea": 380, "Pitesti": 100, "Rimnicu Vilcea": 193, "Sibiu": 253, "Timisoara": 329,
    "Urziceni": 80, "Vaslui": 199, "Zerind": 374,
}

VIZINHOS = {}
for cidade in CIDADES:
    VIZINHOS[cidade] = []
for (a, b), km in ESTRADAS.items():
    VIZINHOS[a].append((b, km))
    VIZINHOS[b].append((a, km))


def h(cidade, destino):
    if destino == "Bucareste":
        return LINHA_RETA_ATE_BUCARESTE[cidade]
    return 0


def a_estrela(origem, destino):
    fronteira = [(h(origem, destino), 0, origem, [origem])]
    visitadas = []
    while len(fronteira) > 0:
        melhor = min(fronteira)
        fronteira.remove(melhor)
        f, g, cidade, caminho = melhor
        if cidade == destino:
            return caminho, g
        if cidade in visitadas:
            continue
        visitadas.append(cidade)
        for vizinha, km in VIZINHOS[cidade]:
            if vizinha not in visitadas:
                novo_g = g + km
                fronteira.append((novo_g + h(vizinha, destino), novo_g, vizinha, caminho + [vizinha]))
    return None, None


print(a_estrela("Arad", "Bucareste"))

## O que é uma ferramenta

Uma **ferramenta** é uma função Python comum, com duas coisas a mais:

1. **Dicas de tipo** nos parâmetros: `origem: str` diz que a origem é um texto.
2. Uma **descrição** (o texto entre as aspas triplas logo abaixo do `def`), com uma frase sobre o que a função faz e uma linha para cada parâmetro depois de `Args:`.

O modelo **nunca vê o código** da função. Ele só lê o **nome**, a **descrição** e os **parâmetros**. É com isso que ele decide quando usar a ferramenta.

Repare também que as ferramentas **conferem os dados** antes de calcular (Laboratório 8) e devolvem **texto**, que o modelo consegue ler.

In [ ]:
def calcular_rota(origem: str, destino: str) -> str:
    """
    Calcula o caminho de menor custo entre duas cidades do mapa da Romênia, usando o algoritmo A*.

    Args:
        origem: cidade de partida, escrita exatamente como na lista de cidades
        destino: cidade de chegada, escrita exatamente como na lista de cidades
    """
    if origem not in CIDADES:
        return "Erro: a cidade " + str(origem) + " não existe no mapa."
    if destino not in CIDADES:
        return "Erro: a cidade " + str(destino) + " não existe no mapa."
    caminho, custo = a_estrela(origem, destino)
    return "Caminho: " + " -> ".join(caminho) + ". Custo total: " + str(custo) + " km."


def listar_vizinhas(cidade: str) -> str:
    """
    Lista as cidades vizinhas de uma cidade do mapa da Romênia e a distância até cada uma.

    Args:
        cidade: a cidade consultada, escrita exatamente como na lista de cidades
    """
    if cidade not in CIDADES:
        return "Erro: a cidade " + str(cidade) + " não existe no mapa."
    partes = []
    for vizinha, km in VIZINHOS[cidade]:
        partes.append(vizinha + " (" + str(km) + " km)")
    return "Vizinhas de " + cidade + ": " + ", ".join(partes) + "."


FERRAMENTAS = [calcular_rota, listar_vizinhas]

print(calcular_rota("Arad", "Bucareste"))
print(listar_vizinhas("Sibiu"))
print(calcular_rota("Arad", "Paris"))

### O que o modelo enxerga

A biblioteca `transformers` transforma cada função em uma **ficha** (em JSON) com o nome, a descrição e os parâmetros. É essa ficha que vai dentro do prompt:

In [ ]:
from transformers.utils import get_json_schema

print(json.dumps(get_json_schema(calcular_rota), indent=1, ensure_ascii=False))

---
# Etapa 2 — O modelo pedindo uma ferramenta

## 🛠️ TODO 1 — As regras do agente

Escreva as regras (mensagem **system**) do agente. Elas precisam dizer:

1. o que o agente faz (ajuda com rotas no mapa da Romênia);
2. que ele deve **usar as ferramentas** para rotas, distâncias e vizinhas;
3. que ele **não deve inventar** distâncias nem caminhos;
4. que a resposta final deve ser **em português**, curta.

In [ ]:
# TODO 1: escreva as regras do agente
REGRAS = ""

print(REGRAS)

In [ ]:
# --- Verificação do TODO 1 (não altere) ---
texto = REGRAS.lower()
falta = []
if "ferramenta" not in texto:
    falta.append("mandar usar as ferramentas")
if "invent" not in texto and "apenas" not in texto and "somente" not in texto:
    falta.append("proibir inventar dados")
if "portugu" not in texto:
    falta.append("pedir resposta em português")
if len(falta) == 0:
    print("✅ TODO 1 correto.")
else:
    print("❌ Ainda não. Falta:", ", ".join(falta))

## O pedido de ferramenta

Quando o modelo decide usar uma ferramenta, ele **não executa nada**. Ele só escreve um texto especial, entre as marcas `<tool_call>` e `</tool_call>`, dizendo qual ferramenta quer e com quais argumentos. Rode e veja:

In [ ]:
mensagens = [
    {"role": "system", "content": REGRAS},
    {"role": "user", "content": "Qual o caminho mais curto de Arad até Bucareste?"},
]
resposta = conversar(mensagens, FERRAMENTAS)
print(resposta)

In [ ]:
mensagens = [
    {"role": "system", "content": REGRAS},
    {"role": "user", "content": "Oi! O que você sabe fazer?"},
]
print(conversar(mensagens, FERRAMENTAS))

**Compare as duas respostas.** Na primeira, o modelo pediu a ferramenta `calcular_rota`. Na segunda, respondeu direto: não havia nada para calcular. **Quem decidiu foi o modelo.**

---
# Etapa 3 — Ler o pedido e executar a ferramenta

## 🛠️ TODO 2 — Ler o pedido de ferramenta

O pedido vem como texto. Por exemplo:

```
<tool_call>
{"name": "calcular_rota", "arguments": {"origem": "Arad", "destino": "Bucareste"}}
</tool_call>
```

A função `extrair_json` é a mesma do Laboratório 8. **Falta uma linha:** se a resposta tem a marca `<tool_call>`, devolva o JSON recortado dela.

*Dica:* `return extrair_json(resposta)`

In [ ]:
def extrair_json(texto):
    inicio = texto.find("{")
    fim = texto.rfind("}")
    if inicio == -1 or fim == -1:
        return None
    try:
        return json.loads(texto[inicio:fim + 1])
    except:
        return None


def ler_pedido_de_ferramenta(resposta):
    if "<tool_call>" not in resposta:
        return None              # o modelo não pediu ferramenta: é a resposta final
    # TODO 2: devolva o JSON recortado da resposta
    return None   # apague esta linha e escreva a sua


exemplo = '<tool_call>\n{"name": "calcular_rota", "arguments": {"origem": "Arad", "destino": "Bucareste"}}\n</tool_call>'
print(ler_pedido_de_ferramenta(exemplo))

In [ ]:
# --- Verificação do TODO 2 (não altere) ---
p1 = ler_pedido_de_ferramenta('<tool_call>\n{"name": "calcular_rota", "arguments": {"origem": "Arad", "destino": "Bucareste"}}\n</tool_call>')
p2 = ler_pedido_de_ferramenta('<tool_call>\n{"name": "listar_vizinhas", "arguments": {"cidade": "Sibiu"}}\n</tool_call>')
p3 = ler_pedido_de_ferramenta("Olá! Eu calculo rotas no mapa da Romênia.")
if p1 is not None and p1["name"] == "calcular_rota" and p2 is not None and p2["arguments"] == {"cidade": "Sibiu"} and p3 is None:
    print("✅ TODO 2 correto. Pedido de ferramenta lido; resposta comum devolve None.")
else:
    print("❌ Ainda não. Confira a linha do TODO 2.")

## 🛠️ TODO 3 — Executar a ferramenta pedida

A função `executar` recebe o pedido lido e chama a função Python certa. O caso de `calcular_rota` já está pronto. **Complete o caso de `listar_vizinhas`.**

*Dica:* `return listar_vizinhas(argumentos["cidade"])`

Se o modelo pedir uma ferramenta que não existe, devolvemos uma mensagem de erro. O erro **volta para o modelo**, que pode tentar de novo ou avisar o usuário.

In [ ]:
def executar(pedido):
    nome = pedido.get("name")
    argumentos = pedido.get("arguments", {})

    if nome == "calcular_rota":
        return calcular_rota(argumentos.get("origem"), argumentos.get("destino"))
    if nome == "listar_vizinhas":
        # TODO 3: chame listar_vizinhas com a cidade dos argumentos
        return None   # apague esta linha e escreva a sua

    return "Erro: a ferramenta " + str(nome) + " não existe."


print(executar({"name": "calcular_rota", "arguments": {"origem": "Oradea", "destino": "Sibiu"}}))
print(executar({"name": "listar_vizinhas", "arguments": {"cidade": "Bucareste"}}))
print(executar({"name": "voar", "arguments": {}}))

In [ ]:
# --- Verificação do TODO 3 (não altere) ---
r1 = executar({"name": "calcular_rota", "arguments": {"origem": "Arad", "destino": "Bucareste"}})
r2 = executar({"name": "listar_vizinhas", "arguments": {"cidade": "Sibiu"}})
r3 = executar({"name": "listar_vizinhas", "arguments": {"cidade": "Iasi"}})
if "418" in r1 and r2 is not None and "Fagaras" in r2 and r3 is not None and "Neamt" in r3:
    print("✅ TODO 3 correto. As duas ferramentas executam.")
else:
    print("❌ Ainda não. listar_vizinhas('Sibiu') deveria trazer Fagaras. Saiu:", r2)

---
# Etapa 4 — O ciclo completo do agente

O agente repete um ciclo curto:

1. **Pensar:** o modelo lê a conversa e decide se precisa de uma ferramenta.
2. **Agir:** se pediu, o código executa a ferramenta.
3. **Observar:** o resultado volta para o modelo como uma mensagem com o papel `"tool"`.
4. **Responder:** quando não pede mais ferramenta, a resposta dele é a final.

Para não girar para sempre, o ciclo tem um **limite de 3 passos**.

## 🛠️ TODO 4 — Devolver o resultado ao modelo

A função `agente` já está montada. **Faltam duas linhas:** acrescentar na conversa (1) a resposta do modelo, com o papel `"assistant"`, e (2) o resultado da ferramenta, com o papel `"tool"`.

*Dica:*
```python
mensagens.append({"role": "assistant", "content": resposta})
mensagens.append({"role": "tool", "content": resultado})
```

In [ ]:
def agente(pergunta, mostrar=True):
    mensagens = [
        {"role": "system", "content": REGRAS},
        {"role": "user", "content": pergunta},
    ]
    for passo in range(3):
        resposta = conversar(mensagens, FERRAMENTAS)       # 1. pensar
        pedido = ler_pedido_de_ferramenta(resposta)
        if pedido is None:
            return resposta                                # 4. responder
        resultado = executar(pedido)                       # 2. agir
        if mostrar:
            print("   [ferramenta]", pedido.get("name"), pedido.get("arguments"))
            print("   [resultado ]", resultado)
        # 3. observar — TODO 4: acrescente as duas mensagens na conversa
        pass   # apague esta linha e escreva as duas linhas

    return "Não consegui concluir em 3 passos."


print(agente("Qual o caminho mais curto de Arad até Bucareste?"))

In [ ]:
# --- Verificação do TODO 4 (não altere) ---
final = agente("Quero ir de Oradea para Sibiu. Qual a rota?", mostrar=False)
if "151" in final:
    print("✅ TODO 4 correto. O modelo recebeu o resultado do A* e explicou: ")
    print("  ", final)
else:
    print("❌ Ainda não. O resultado da ferramenta (151 km) não chegou à resposta final.")
    print("   Resposta:", final)

## Testando o agente

Rode e observe, para cada pergunta: o modelo usou ferramenta? Qual? A resposta final usou os números da ferramenta ou inventou?

In [ ]:
perguntas = [
    "Quais são as cidades vizinhas de Sibiu?",
    "preciso chegar em sibiu, estou em oradea",
    "Oi! O que você sabe fazer?",
    "Me leva de Arad para Paris.",
]
for pergunta in perguntas:
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", agente(pergunta))
    print()

---
# Etapa 5 — Testar a escolha da ferramenta

Na Aula 11 vamos montar os 15 casos de teste do Marco 2. Hoje medimos uma coisa só: o modelo **escolhe a ferramenta certa**?

Cada caso tem a pergunta e a ferramenta esperada (`None` quando não deve usar nenhuma).

## 🛠️ TODO 5 — Acrescente dois casos

Um em que a ferramenta certa é `listar_vizinhas` e um em que **nenhuma** ferramenta deve ser usada.

In [ ]:
CASOS = [
    ("Qual o caminho mais curto de Arad até Bucareste?", "calcular_rota"),
    ("Como chego em Pitesti saindo de Timisoara?", "calcular_rota"),
    ("Quais cidades ficam ao lado de Craiova?", "listar_vizinhas"),
    ("Bom dia!", None),
    # TODO 5: acrescente dois casos (pergunta, ferramenta esperada)
]

print(len(CASOS), "casos")

In [ ]:
# --- Verificação do TODO 5 (não altere) ---
esperadas = []
for pergunta, ferramenta in CASOS:
    esperadas.append(ferramenta)
if len(CASOS) >= 6 and esperadas.count("listar_vizinhas") >= 2 and esperadas.count(None) >= 2:
    print("✅ TODO 5 correto.", len(CASOS), "casos.")
else:
    print("❌ Ainda não. Precisa de pelo menos 6 casos, com 2 de listar_vizinhas e 2 sem ferramenta.")

In [ ]:
# Mede só a PRIMEIRA decisão do modelo: qual ferramenta ele pediu.
acertos = 0
for pergunta, esperada in CASOS:
    mensagens = [{"role": "system", "content": REGRAS}, {"role": "user", "content": pergunta}]
    pedido = ler_pedido_de_ferramenta(conversar(mensagens, FERRAMENTAS))
    if pedido is None:
        escolhida = None
    else:
        escolhida = pedido.get("name")
    if escolhida == esperada:
        acertos = acertos + 1
        marca = "certo "
    else:
        marca = "ERRADO"
    print(marca, "|", pergunta, "| esperada:", esperada, "| escolhida:", escolhida)

print()
print("Escolheu a ferramenta certa em", acertos, "de", len(CASOS), "casos.")

---
# Desafio Final (Entrega Obrigatória) — A ferramenta do seu trio

Agora o agente passa a usar o **A\* do seu trio**.

## 🛠️ TODO 6 — Transforme o A\* do trio em ferramenta

1. Cole na célula abaixo o **A\* do Marco 1**, como função `a_estrela_trio(origem, destino)` que devolve `(caminho, custo)`. Se ainda não estiver pronto, use o exemplo que já está lá.
2. Preencha `ESTADOS_DO_TRIO`.
3. Escreva a **descrição** da ferramenta `ferramenta_do_trio`: uma frase dizendo o que ela faz **no domínio do trio** e uma linha para cada parâmetro.

In [ ]:
# 1) o A* do Marco 1 do trio (troque o exemplo pelo de vocês)
def a_estrela_trio(origem, destino):
    return a_estrela(origem, destino)

# 2) os estados válidos do domínio do trio
ESTADOS_DO_TRIO = CIDADES

# 3) a ferramenta: escreva a descrição entre as aspas triplas
def ferramenta_do_trio(origem: str, destino: str) -> str:
    """
    ESCREVA AQUI O QUE A FERRAMENTA FAZ NO DOMÍNIO DO TRIO.

    Args:
        origem: ESCREVA AQUI
        destino: ESCREVA AQUI
    """
    if origem not in ESTADOS_DO_TRIO or destino not in ESTADOS_DO_TRIO:
        return "Erro: estado fora da lista. Estados válidos: " + ", ".join(ESTADOS_DO_TRIO)
    caminho, custo = a_estrela_trio(origem, destino)
    if caminho is None:
        return "Não existe caminho entre " + origem + " e " + destino + "."
    return "Caminho: " + " -> ".join(caminho) + ". Custo total: " + str(custo) + "."


print(ferramenta_do_trio(ESTADOS_DO_TRIO[0], ESTADOS_DO_TRIO[-1]))

In [ ]:
# --- Verificação do TODO 6 (não altere) ---
descricao = ferramenta_do_trio.__doc__ or ""
if "ESCREVA AQUI" not in descricao and "Args:" in descricao and len(descricao.strip()) > 60:
    ficha = get_json_schema(ferramenta_do_trio)
    print("✅ TODO 6 correto. É isto que o modelo vai ler sobre a ferramenta do trio:")
    print(json.dumps(ficha["function"], indent=1, ensure_ascii=False))
else:
    print("❌ Ainda não. Troque todos os 'ESCREVA AQUI' da descrição.")

## O agente do trio

Ajuste as regras e as perguntas para o domínio do trio e rode. Escolha **três perguntas**: uma fácil, uma difícil e uma que **não** precisa de ferramenta.

In [ ]:
REGRAS_DO_TRIO = REGRAS     # troque pelas regras do domínio do trio
FERRAMENTAS = [ferramenta_do_trio]

perguntas_do_trio = [
    "pergunta fácil do domínio do trio",
    "pergunta difícil do domínio do trio",
    "pergunta que não precisa de ferramenta",
]

REGRAS = REGRAS_DO_TRIO
for pergunta in perguntas_do_trio:
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", agente(pergunta))
    print()

### Sua análise — edite esta célula

Responda em 4 a 6 linhas:

1. **A escolha:** em quantos dos casos da Etapa 5 o modelo escolheu a ferramenta certa? Em qual errou?
2. **A explicação:** em alguma resposta final o modelo escreveu um número que **não** veio da ferramenta?
3. **O seminário:** na demonstração ao vivo vocês precisam mostrar **um caso de falha** do agente. Qual pergunta do domínio do trio faz o agente falhar, e por quê?

---

_Escreva sua análise aqui, substituindo esta linha._

---

---
## ✅ Checklist antes de entregar

- [ ] Os seis TODOs com ✅ na verificação.
- [ ] O token do Hugging Face está nos **Segredos** (🔑) e **não aparece** em nenhuma célula.
- [ ] O Desafio Final usa o A\* e os estados **do seu trio**.
- [ ] A célula **Sua análise** foi preenchida, com um caso de falha para o seminário.
- [ ] `Ambiente de execução → Reiniciar sessão e executar tudo` roda sem erro.

## Resumo de fixação

| Conceito da aula | Onde apareceu no notebook |
|---|---|
| Ferramenta = função + descrição | `calcular_rota`, `listar_vizinhas` e `get_json_schema` |
| O modelo pede, não executa | `<tool_call>` e `ler_pedido_de_ferramenta` (TODO 2) |
| O código confere e executa | `executar` (TODO 3) e a conferência dentro das ferramentas |
| Pensar → agir → observar → responder | `agente` (TODO 4) e o limite de 3 passos |
| Testar a escolha da ferramenta | `CASOS` (TODO 5) |
| O A\* do trio como ferramenta | `ferramenta_do_trio` (TODO 6) |

---
**Para a próxima aula (04/11 — Avaliação e IA responsável):** tragam o agente do trio funcionando com o A\* do Marco 1 e uma lista de **15 perguntas** do domínio, com a resposta esperada de cada uma.